# Gemma 4 31B — Demo Notebook

Walks through the main capabilities of the served model:

1. Setup
2. Text inference
3. Reasoning / thinking mode
4. Tool calling
5. Image understanding
6. Document OCR

**Prerequisites:** server running on the instance (`bash serve.sh`), `.env` configured with `VLLM_BASE_URL`.

## 1 · Setup

In [1]:
import base64, json, textwrap
from pathlib import Path
from IPython.display import display, Markdown

from dotenv import load_dotenv
load_dotenv()

from src.client import get_client, MODEL_ID

client = get_client()

# Confirm the server is reachable
models = client.models.list()
print("Server is up. Available models:")
for m in models.data:
    print(" ", m.id)

Server is up. Available models:
  RedHatAI/gemma-4-31B-it-FP8-block


## 2 · Text Inference

In [2]:
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": "Explain the difference between supervised and unsupervised learning in 3 short paragraphs."}
    ],
    max_tokens=8192,
    temperature=0.7,
)

display(Markdown(response.choices[0].message.content))

**Supervised learning** is a process where a model is trained on a labeled dataset. This means the input data is already paired with the correct output (the "answer key"). The goal is for the algorithm to learn a mapping function that can predict the label of new, unseen data based on the patterns it identified during training. Common examples include spam detection and house price prediction.

**Unsupervised learning**, by contrast, works with unlabeled data. The algorithm is given a set of inputs without any explicit instructions on what the output should be. Instead of predicting a specific value, the model looks for inherent structures, patterns, or anomalies within the data on its own. A common application is customer segmentation, where a business groups customers by similar buying habits.

The fundamental difference lies in the **presence of a teacher**. In supervised learning, the labels act as a teacher guiding the model toward the correct answer. In unsupervised learning, the model acts as an explorer, discovering hidden relationships within the data without any external guidance or predefined categories.

## 3 · Reasoning / Thinking Mode

The model reasons step-by-step before producing its final answer. The thinking trace is returned in `message.reasoning`.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": (
                "I have a 3-litre jug and a 5-litre jug with no markings. "
                "How do I measure exactly 4 litres of water?"
            )
        }
    ],
    max_tokens=8192,
)

message = response.choices[0].message

if getattr(message, "reasoning", None):
    display(Markdown("### Thinking\n" + message.reasoning))

display(Markdown("### Answer\n" + message.content))

## 4 · Tool Calling

The model can call functions you define. Here we give it a weather tool and let it decide when and how to use it.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name, e.g. 'Tokyo'"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather like in Tokyo and Sydney right now?"}]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=tools,
    max_tokens=8192,
)

assistant_msg = response.choices[0].message
messages.append(assistant_msg)

print("Tool calls made:")
for tc in assistant_msg.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")

    # Simulate tool responses
    fake_results = {
        "Tokyo": {"temp": 22, "condition": "Partly cloudy", "unit": "celsius"},
        "Sydney": {"temp": 18, "condition": "Sunny", "unit": "celsius"},
    }
    args = json.loads(tc.function.arguments)
    result = fake_results.get(args["location"], {"temp": 20, "condition": "Unknown"})

    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps(result),
    })

# Final answer after tool results
followup = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=tools,
    max_tokens=8192,
)

display(Markdown("### Final answer\n" + followup.choices[0].message.content))

## 5 · Image Understanding

Pass any image URL or local file. The cell below fetches a public image and asks the model to describe it.

In [ ]:
from IPython.display import Image as IPImage

image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"

# Show the image inline
display(IPImage(url=image_url, width=400))

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": "Describe this image in detail. What breed might this be, and what is the animal doing?"},
            ],
        }
    ],
    max_tokens=8192,
    temperature=0.0,
)

display(Markdown(response.choices[0].message.content))

In [ ]:
# Alternatively — upload a local image file
# Change this path to any image on your machine
LOCAL_IMAGE = None  # e.g. "/path/to/document.png"

if LOCAL_IMAGE:
    data = Path(LOCAL_IMAGE).read_bytes()
    suffix = Path(LOCAL_IMAGE).suffix.lstrip(".").lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png"}.get(suffix, "image/jpeg")
    data_url = f"data:{mime};base64,{base64.b64encode(data).decode()}"

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": data_url}},
                    {"type": "text", "text": "Describe this image in detail."},
                ],
            }
        ],
        max_tokens=8192,
        temperature=0.0,
    )
    display(Markdown(response.choices[0].message.content))
else:
    print("Set LOCAL_IMAGE to a file path to use this cell.")

## 6 · Document OCR

Each page is sent as a separate request using vLLM's guided decoding to produce a structured `DocumentOCR` object.

Supported inputs: PDF, PNG, JPEG, WEBP.

In [ ]:
from src.ocr import ocr_document

# Change this to a PDF or image path on your machine
DOCUMENT_PATH = None  # e.g. "/path/to/document.pdf"

if DOCUMENT_PATH:
    result = ocr_document(DOCUMENT_PATH, max_workers=4)

    print(f"Title:    {result.title}")
    print(f"Language: {result.language}")
    print(f"Pages:    {result.page_count}")
    print()

    for i, page in enumerate(result.pages, 1):
        display(Markdown(f"### Page {i} — {len(page.blocks)} blocks"))
        rows = [[f"`{b.type.value}`", b.text[:120] + ("…" if len(b.text) > 120 else "")] for b in page.blocks]
        table = "| Type | Text |\n|------|------|\n" + "\n".join(f"| {r[0]} | {r[1]} |" for r in rows)
        display(Markdown(table))
else:
    print("Set DOCUMENT_PATH to a PDF or image file to run OCR.")

### OCR on an image URL (no local file needed)

In [ ]:
import urllib.request
from src.ocr import ocr_page

# A sample document image from Wikipedia
doc_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/31/USDC_Complaint_p1.jpg/800px-USDC_Complaint_p1.jpg"

display(IPImage(url=doc_url, width=500))

image_bytes = urllib.request.urlopen(doc_url).read()
page_result = ocr_page(image_bytes, suffix="jpg")

rows = [[f"`{b.type.value}`", b.text[:120] + ("…" if len(b.text) > 120 else "")] for b in page_result.blocks]
table = "| Type | Text |\n|------|------|\n" + "\n".join(f"| {r[0]} | {r[1]} |" for r in rows)
display(Markdown(table))